In [37]:
import numpy as np
import os
from astropy.table import Table, vstack
# (Не обов'язково, але рекомендовано для ECSV:)
try:
    from astropy.table import QTable
    import astropy.units as u
    has_astropy = True
except Exception:
    has_astropy = False
    import astropy.units as u
import numpy as np
import naima
from astropy.constants import c
from astropy.io import ascii
import os
from naima.models import PowerLaw, InverseCompton, PionDecay
from astropy.table import Table

In [29]:
F_int = 4.3e-13        # cm^-2 s^-1 (integral > 1 TeV)
Gamma = 2.3
Gamma_err = 0.2
F_rel_err = 0.2        # δF = 0.2 F
E0_GeV = 1000.0 

In [41]:
E_full = np.logspace(np.log10(1000.0), np.log10(10000.0), 400)

In [42]:
N0_mean = F_int * (Gamma - 1.0) / E0_GeV
norm_min = N0_mean * (1.0 - F_rel_err)
norm_max = N0_mean * (1.0 + F_rel_err)
index_min = Gamma - Gamma_err
index_max = Gamma + Gamma_err

norms = np.linspace(norm_min, norm_max, 200)
indices = np.linspace(index_min, index_max, 200)
norm_grid, index_grid = np.meshgrid(norms, indices, indexing='ij')

E_b = E_full[:, np.newaxis, np.newaxis]
N0_b = norm_grid[np.newaxis, :, :]
Gamma_b = index_grid[np.newaxis, :, :]

dNdE_grid = N0_b * (E_b / E0_GeV) ** (-Gamma_b)

In [43]:
y_min = np.min(dNdE_grid, axis=(1,2))
y_max = np.max(dNdE_grid, axis=(1,2))
y_mean =  N0_mean * (E_full / E0_GeV)**(-Gamma)

In [44]:
N_bins = 20 # Нова змінна: бажана кількість логарифмічних бінів
E_min_export = 1000.0
E_max_export = 10000.0

bin_edges_log = np.linspace(np.log10(E_min_export), np.log10(E_max_export), N_bins + 1)

E_export_log = (bin_edges_log[:-1] + bin_edges_log[1:]) / 2.0


In [45]:
E_export = 10**E_export_log         
E_low_export = 10**bin_edges_log[:-1] 
E_high_export = 10**bin_edges_log[1:]

In [46]:
y_mean_interp = np.interp(np.log10(E_export), np.log10(E_full), y_mean)
y_min_interp  = np.interp(np.log10(E_export), np.log10(E_full), y_min)
y_max_interp  = np.interp(np.log10(E_export), np.log10(E_full), y_max)

err_low = y_mean_interp - y_min_interp
err_high = y_max_interp - y_mean_interp

err_low = np.maximum(err_low, 0.0)
err_high = np.maximum(err_high, 0.0)

#error_export = np.minimum(err_low, err_high)

In [47]:
data = QTable([
    (E_export * u.GeV),  # Конвертуємо e_ctr з MeV в GeV
    (E_low_export * u.GeV),  # Конвертуємо e_min з MeV в GeV
    (E_high_export * u.GeV),  # Конвертуємо e_max з MeV в GeV
    (y_mean_interp * u.Unit('1 / (cm2 s GeV)')), # Конвертуємо потік
    #(error_export * u.Unit('1 / (cm2 s GeV)')) # Конвертуємо похибку
    (err_low * u.Unit('1 / (cm2 s GeV)')),
    (err_high * u.Unit('1 / (cm2 s GeV)'))
],
names=("energy", "energy_edge_lo", "energy_edge_hi", "flux", "flux_error_lo", "flux_error_hi"))
data.meta["cl"] = 0.95

output_filename = 'HESS_10_TeV.dat'
data.write(output_filename, format='ipac', overwrite=True)

print(f"Дані успішно збережено у файл: {output_filename}")

Дані успішно збережено у файл: HESS_10_TeV.dat


In [48]:
data_lat = ascii.read('SED2.dat', format='ipac')
data_hess = ascii.read('HESS_10_TeV.dat', format = 'ipac')

In [49]:
data_lat['flux_error_lo'] = data_lat['flux_error']
data_lat['flux_error_hi'] = data_lat['flux_error']

In [50]:
data_lat.remove_column('flux_error')

In [51]:
combined_data = vstack([data_lat, data_hess])

In [52]:
output_filename = 'SED2_HESS_10_TeV.txt'
combined_data.write(output_filename, format='ipac', overwrite=True)